In [1]:
# experiments/05_generalization.py

import torch
import sys
import copy
import statistics
from datasets import load_dataset
from transformers import (AutoTokenizer,
                          BertForSequenceClassification,
                          BertConfig)
from torchao.quantization.qat.linear import Int8DynActInt4WeightQATLinear
import torch._dynamo

sys.path.insert(0, '/home/shreya/Coding/optimizer_V2')
from src.graph.boundary_detector import ActivationStabilityDetector
from src.graph.static_converter import StaticScaleLinear, convert_stable_layers

torch._dynamo.config.cache_size_limit = 64
device    = torch.device('cuda')
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# ── same apply_qat function, works for any model ─────────────
def apply_qat_to_linear(model):
    for name, module in model.named_children():
        if isinstance(module, torch.nn.Linear):
            qat_linear = Int8DynActInt4WeightQATLinear(
                module.in_features,
                module.out_features,
                bias=False,
                groupsize=32,
            )
            with torch.no_grad():
                qat_linear.weight.copy_(module.weight)
            if module.bias is not None:
                qat_linear.bias = torch.nn.Parameter(module.bias.clone())
            setattr(model, name, qat_linear)
        else:
            apply_qat_to_linear(module)
    return model


/home/shreya/venvs/fusion_qat/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# ── data ────────────────────────────────────────────────────
dataset_train = load_dataset("glue", "sst2", split="train[:5000]")
dataset_val   = load_dataset("glue", "sst2", split="validation")

def encode_and_batch(dataset, batch_size=16):
    sentences = list(dataset['sentence'])
    labels    = list(dataset['label'])
    enc  = tokenizer(sentences, padding='max_length',
                     truncation=True, max_length=64)
    ids   = torch.tensor(enc['input_ids'])
    masks = torch.tensor(enc['attention_mask'])
    labs  = torch.tensor(labels)
    batches = []
    for i in range(0, len(ids) - batch_size, batch_size):
        batches.append({
            'input_ids':      ids[i:i+batch_size],
            'attention_mask': masks[i:i+batch_size],
            'labels':         labs[i:i+batch_size]
        })
    return batches

train_batches = encode_and_batch(dataset_train, batch_size=16)
val_batches   = encode_and_batch(dataset_val,   batch_size=16)

# ── build and fine-tune BERT-base QAT ────────────────────────
bert_model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=2
).to(device)

bert_qat = apply_qat_to_linear(bert_model)
bert_qat = bert_qat.to(device)

qat_count = sum(1 for _, m in bert_qat.named_modules()
                if isinstance(m, Int8DynActInt4WeightQATLinear))
print(f"BERT-base QAT layers: {qat_count}")


Loading weights: 100%|█████████████████████| 199/199 [00:00<00:00, 14765.02it/s]
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint.

BERT-base QAT layers: 74


In [3]:

# fine-tune
optimizer = torch.optim.AdamW(bert_qat.parameters(), lr=2e-5)
criterion = torch.nn.CrossEntropyLoss()

for epoch in range(3):
    bert_qat.train()
    losses = []
    for batch in train_batches:
        inputs = {k: v.to(device) for k, v in batch.items()
                  if k != 'labels'}
        labels = batch['labels'].to(device)
        optimizer.zero_grad()
        out  = bert_qat(**inputs)
        loss = criterion(out.logits, labels)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    bert_qat.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in val_batches:
            inputs = {k: v.to(device) for k, v in batch.items()
                      if k != 'labels'}
            labels = batch['labels'].to(device)
            out    = bert_qat(**inputs)
            correct += (out.logits.argmax(-1) == labels).sum().item()
            total   += labels.size(0)
    acc = correct / total * 100
    print(f"Epoch {epoch+1}: loss={statistics.mean(losses):.4f}  "
          f"val_acc={acc:.2f}%")


Epoch 1: loss=0.3881  val_acc=81.83%
Epoch 2: loss=0.1809  val_acc=86.46%
Epoch 3: loss=0.0891  val_acc=86.69%


In [4]:

# ── measure latency ───────────────────────────────────────────
def measure_latency(model, warmup=5, runs=30):
    compiled = torch.compile(model, backend="inductor")
    model.eval()
    with torch.no_grad():
        for batch in val_batches[:warmup]:
            inputs = {k: v.to(device) for k, v in batch.items()
                      if k != 'labels'}
            compiled(**inputs)
    torch.cuda.synchronize()
    times = []
    with torch.no_grad():
        for batch in val_batches[:runs]:
            inputs = {k: v.to(device) for k, v in batch.items()
                      if k != 'labels'}
            start = torch.cuda.Event(enable_timing=True)
            end   = torch.cuda.Event(enable_timing=True)
            start.record()
            compiled(**inputs)
            end.record()
            torch.cuda.synchronize()
            times.append(start.elapsed_time(end))
    return statistics.mean(times), statistics.stdev(times)

lat_qat, std_qat = measure_latency(bert_qat)
print(f"\nBERT-base QAT latency: {lat_qat:.1f}ms ± {std_qat:.1f}ms")

# ── run stability analysis ────────────────────────────────────
detector = ActivationStabilityDetector(cv_threshold_static=5.0)
detector.attach_hooks(bert_qat)

bert_qat.eval()
with torch.no_grad():
    for batch in val_batches:
        inputs = {k: v.to(device) for k, v in batch.items()
                  if k != 'labels'}
        bert_qat(**inputs)

detector.remove_hooks()
labels_bert = detector.compute_labels()
detector.summary(labels_bert)

# ── convert and measure ───────────────────────────────────────
bert_converted, report = convert_stable_layers(
    copy.deepcopy(bert_qat), labels_bert, include_borderline=False
)

static_count = sum(1 for _, m in bert_converted.named_modules()
                   if isinstance(m, StaticScaleLinear))
total_layers = qat_count
print(f"BERT-base converted: {static_count}/{total_layers} layers")

bert_converted.eval()
correct = total = 0
with torch.no_grad():
    for batch in val_batches:
        inputs = {k: v.to(device) for k, v in batch.items()
                  if k != 'labels'}
        labels = batch['labels'].to(device)
        out    = bert_converted(**inputs)
        correct += (out.logits.argmax(-1) == labels).sum().item()
        total   += labels.size(0)
acc_converted = correct / total * 100

lat_conv, std_conv = measure_latency(bert_converted)

print(f"\n{'='*60}")
print(f"  BERT-BASE GENERALIZATION RESULTS")
print(f"{'='*60}")
print(f"  {'Model':<30} {'Latency':>10}  {'Accuracy':>10}")
print(f"  {'-'*52}")
print(f"  {'BERT-base QAT dynamic':<30} "
      f"{f'{lat_qat:.1f}ms':>10}  {f'{acc:.2f}%':>10}")
print(f"  {'BERT-base static converted':<30} "
      f"{f'{lat_conv:.1f}ms':>10}  {f'{acc_converted:.2f}%':>10}")
print(f"  {'-'*52}")
print(f"  Speedup:        {lat_qat/lat_conv:.2f}x")
print(f"  Accuracy cost:  {acc - acc_converted:.2f}%")
print(f"  Layers static:  {static_count}/{total_layers}")
print(f"{'='*60}")

/home/shreya/venvs/fusion_qat/lib/python3.12/site-packages/torch/_inductor/compile_fx.py:194: UserWarning: TensorFloat32 tensor cores for float32 matrix multiplication available but not enabled. Consider setting `torch.set_float32_matmul_precision('high')` for better performance.
  warnings.warn(
W0422 11:43:03.167000 90438 torch/_inductor/utils.py:1137] [15/4] Not enough SMs to use max_autotune_gemm mode
W0422 11:43:05.627000 90438 torch/_dynamo/convert_frame.py:906] [25/64] torch._dynamo hit config.cache_size_limit (64)
W0422 11:43:05.627000 90438 torch/_dynamo/convert_frame.py:906] [25/64]    function: 'apply_chunking_to_forward' (/home/shreya/venvs/fusion_qat/lib/python3.12/site-packages/transformers/pytorch_utils.py:126)
W0422 11:43:05.627000 90438 torch/_dynamo/convert_frame.py:906] [25/64]    last reason: 25/63: Cache line invalidated because L['forward_fn'] got deallocated
W0422 11:43:05.627000 90438 torch/_dynamo/convert_frame.py:906] [25/64] To log all recompilation reasons, 


BERT-base QAT latency: 79.4ms ± 20.3ms
Attached hooks to 74 QAT layers

  ACTIVATION STABILITY SUMMARY
  STATIC OK  (cv <  5%):  68/74
  BORDERLINE (cv 5-15%):   6/74
  DYNAMIC    (cv > 15%):   0/74

  Estimated savings if STATIC_OK layers frozen:
  choose_qparams + amin eliminated: 68 calls
  Time saved per inference:         23.2ms
  Current QAT compiled latency:     69.6ms
  Projected latency:                46.4ms
BERT-base converted: 68/74 layers

  BERT-BASE GENERALIZATION RESULTS
  Model                             Latency    Accuracy
  ----------------------------------------------------
  BERT-base QAT dynamic              79.4ms      86.69%
  BERT-base static converted         53.0ms      79.75%
  ----------------------------------------------------
  Speedup:        1.50x
  Accuracy cost:  6.94%
  Layers static:  68/74
